# Random Forest Synthetic Feasibility Test
**Purpose:** consolidates the Section 3.6 experiment into a reproducible notebook. Tests whether Random Forest can recover a known, deliberately planted relationship at two sample sizes — n=1000 and n=21 (matching MinoriLabs' real scale).

No real MinoriLabs data is used here — entirely synthetic, by design, since the formula is known in advance and serves as ground truth.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1. Define the Synthetic Data Generator
Matches MinoriLabs' schema (7 input ratio columns). The output is computed from a deliberately chosen formula plus noise.

In [2]:
def generate_synthetic_data(n, seed=None):
    rng = np.random.default_rng(seed)

    eta_achievement = rng.uniform(0, 1.2, n)
    rework_rate = rng.uniform(0, 0.5, n)
    req_understanding_ratio = rng.uniform(0, 0.3, n)
    meeting_overhead_ratio = rng.uniform(0, 0.2, n)
    idle_rate = rng.uniform(0, 0.4, n)
    effective_utilisation = rng.uniform(0, 1.2, n)
    avg_time_per_task = rng.uniform(0.5, 5.0, n)

    noise = rng.normal(0, 0.15, n)

    output = (
        0.4 * eta_achievement
        - 0.3 * rework_rate
        + 0.2 * effective_utilisation
        + 0.1 * req_understanding_ratio
        + noise
    )

    df = pd.DataFrame({
        "ETA_Achievement": eta_achievement,
        "Rework_Rate": rework_rate,
        "Req_Understanding_Ratio": req_understanding_ratio,
        "Meeting_Overhead_Ratio": meeting_overhead_ratio,
        "Idle_Rate": idle_rate,
        "Effective_Utilisation": effective_utilisation,
        "Avg_Time_Per_Task": avg_time_per_task,
        "Output": output,
    })
    return df


## 2. Test at n=1000 — Establishing a Working Baseline

In [3]:
def run_rf_test(n, seed=RANDOM_SEED):
    df = generate_synthetic_data(n, seed=seed)
    X = df.drop(columns=["Output"])
    y = df["Output"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed
    )

    rf = RandomForestRegressor(n_estimators=100, random_state=seed)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)

    return r2_score(y_test, preds)

r2_n1000 = run_rf_test(1000)
print(f"n=1000: R2 = {r2_n1000:.3f}")


n=1000: R2 = 0.509


## 3. Test at n=21 — Matching MinoriLabs' Real Scale
Run across multiple random draws/seeds, since a single n=21 result could easily be a fluke of that particular sample.

In [4]:
SEEDS_TO_TEST = [1, 7, 42, 99, 2024, 123, 456, 789]

r2_n21_results = [run_rf_test(21, seed=s) for s in SEEDS_TO_TEST]

print("n=21 R2 across different random draws:")
for s, r2 in zip(SEEDS_TO_TEST, r2_n21_results):
    print(f"  seed={s}: R2 = {r2:.3f}")

print(f"\nMean R2 at n=21: {np.mean(r2_n21_results):.3f}")
print(f"Std dev: {np.std(r2_n21_results):.3f}")
print(f"Range: {min(r2_n21_results):.3f} to {max(r2_n21_results):.3f}")


n=21 R2 across different random draws:
  seed=1: R2 = -0.248
  seed=7: R2 = -0.294
  seed=42: R2 = 0.303
  seed=99: R2 = -0.406
  seed=2024: R2 = 0.270
  seed=123: R2 = 0.072
  seed=456: R2 = 0.159
  seed=789: R2 = 0.115

Mean R2 at n=21: -0.004
Std dev: 0.255
Range: -0.406 to 0.303


## 4. Summary

Compare the n=1000 result (a single, stable R2) against the n=21 results (multiple, variable R2 values across different random draws of the same underlying formula). A large spread — including values near zero or negative — demonstrates that sample size, not the absence of a real relationship, is the limiting factor at MinoriLabs' actual scale.

**Reference values from the original thesis analysis:** R2 approx 0.49 at n=1000; R2 unstable and near zero at n=21. Exact values here may differ slightly (different formula coefficients or noise level may have been used originally), but the qualitative pattern — stable, moderate R2 at large n; unstable, near-chance R2 at n=21 — should reproduce.